# NB00 · Desplegar y actualizar el proyecto en el servidor de la cátedra
**Ingeniería de Datos — LCD · UCA Rosario · 2026** · grupo06 · Clima de Sídney (Open-Meteo)

El Airflow compartido **solo monta la carpeta `~/work/dags`** de nuestro workspace
(la ve como `/opt/airflow/dags/grupo06`). Por eso el repositorio completo se clona *dentro*
de `~/work/dags`, y el `.airflowignore` hace que Airflow solo parsee `dags/pipeline_grupo06.py`.

Este notebook:
1. clona o actualiza (`git pull`) el repositorio en `~/work/dags`,
2. genera el `.env` del grupo **a partir de las variables que JupyterHub ya inyecta** (no se escriben claves a mano y el `.env` nunca se versiona),
3. verifica MinIO y PostgreSQL.

In [ ]:
import os, subprocess
REPO = "https://github.com/FMiles-2006/ingdatos-clima-sidney-grupo06.git"
DESTINO = os.path.expanduser("~/work/dags")

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-2000:])
    return r.returncode

if os.path.isdir(os.path.join(DESTINO, ".git")):
    sh(f"cd {DESTINO} && git pull --ff-only")
else:
    # Primera vez: la carpeta puede no estar vacía (Airflow deja __pycache__), por eso init + fetch.
    sh(f"cd {DESTINO} && git init -q -b main && git remote add origin {REPO} && git fetch -q origin main && git checkout -f -B main origin/main")
sh(f"cd {DESTINO} && git log --oneline -5")

## 2. `.env` del grupo (credenciales fuera del repositorio)

In [ ]:
e = os.environ
valores = {
    "GRUPO": "grupo06", "MINIO_BUCKET": e["MINIO_BUCKET"],
    "AWS_ACCESS_KEY_ID": e["AWS_ACCESS_KEY_ID"], "AWS_SECRET_ACCESS_KEY": e["AWS_SECRET_ACCESS_KEY"],
    "AWS_ENDPOINT_URL": e.get("AWS_ENDPOINT_URL", "http://minio:9000"), "AWS_REGION": "us-east-1", "AWS_ALLOW_HTTP": "true",
    "PGHOST": e["PGHOST"], "PGPORT": e["PGPORT"], "PGDATABASE": e["PGDATABASE"],
    "PGUSER": e["PGUSER"], "PGPASSWORD": e["PGPASSWORD"], "PGSCHEMA": "grupo06",
}
with open(os.path.join(DESTINO, ".env"), "w") as f:
    f.write("# Generado por NB00 desde las variables del Jupyter de grupo06. NO versionar.\n")
    f.writelines(f"{k}={v}\n" for k, v in valores.items())
print("Claves escritas en .env:", list(valores))   # solo los nombres, nunca los valores
sh(f"cd {DESTINO} && git check-ignore -v .env")      # confirma que git lo ignora

## 3. Verificar el stack

In [ ]:
import os, sys
from pathlib import Path
# Raíz del proyecto = carpeta padre de notebooks/ (en el servidor: ~/work/dags)
RAIZ = Path(DESTINO)
sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)
import entorno
entorno.info()
import boto3
from sqlalchemy import create_engine, text
so = entorno.STORAGE_OPTIONS
s3 = boto3.client("s3", endpoint_url=so["AWS_ENDPOINT_URL"], aws_access_key_id=so["AWS_ACCESS_KEY_ID"],
                  aws_secret_access_key=so["AWS_SECRET_ACCESS_KEY"], region_name="us-east-1")
r = s3.list_objects_v2(Bucket=entorno.BUCKET, Prefix="bronze/clima_sidney/_delta_log/")
print("MinIO OK · versiones del log de bronze:", r.get("KeyCount", 0))
with create_engine(entorno.pg_url()).connect() as c:
    print("PostgreSQL OK ·", c.execute(text("select current_user, current_database(), current_schema()")).fetchone())